# ASR éwé v0.1 — Notebook 06 : corpus WAXAL et baselines

Projet : **tg_nlp_toolkit** — brique ASR. Contexte, décisions et licences :
`docs/14-asr-ewe-plan-execution-v0.1.md`.

Ce notebook :

1. charge le corpus WAXAL `ewe_asr` (train 15 054 / validation 1 916 / test 1 891 extraits annoncés) ;
2. produit les statistiques du corpus : heures, durées, locuteurs, orthographe, style de transcription ;
3. évalue deux baselines sur le split test :
   - **MMS-1B zero-shot** (Meta, adaptateur `ewe`) — licence CC-BY-NC-4.0, usage évaluation uniquement ;
   - **WAXALNet** `waxal-benchmarking/whisper-small-waxal-ewe` — référence publique (WER 32,3 annoncé sur leur sous-ensemble privé).

**Prérequis** : Google Colab, GPU T4 (Runtime > Change runtime type > T4 GPU).
**Durée estimée** : ~1 h 30 (téléchargement ~5,7 Go, statistiques, puis les deux évaluations). L'évaluation MMS complète est la plus longue : réduire `MAX_EVAL` à ~300 pour un premier passage rapide.

Note technique : la config `ewe_asr` déclarée dans le YAML du dataset est cassée (colonne `__index_level_0__` absente du schéma) ; on lit les parquet directement avec le builder générique. L'audio est du MP3 44,1 kHz stéréo (vérifié 2026-11-16).




In [ ]:
!pip install -q -U datasets soundfile scipy transformers accelerate jiwer




In [ ]:
import io
import json
import re
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import jiwer
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from scipy import signal

warnings.filterwarnings("ignore", message=".*Xing.*")

REPO = "google/WaxalNLP"
LANGUE = "ewe"
TACHE = "ASR"
SPLITS = ["train", "validation", "test"]
SEED = 42

SORTIES = Path("resultats")
SORTIES.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Materiel :", DEVICE, "| torch", torch.__version__)




In [ ]:
# Fonctions communes (egalement dans src/asr/prep_waxal_ewe.py,
# version canonique : manifeste + filtres documentes).

MOTIF_FICHIER = __import__("re").compile(r"^(?P<langue>[a-z]{3})-(?P<split>[a-z]+)-\d+\.parquet$")


def fichiers_parquet(split):
    """Liste triee des parquet d'un split, via l'API Hugging Face."""
    prefixe = f"data/{TACHE}/{LANGUE}/"
    noms = [
        f for f in HfApi().list_repo_files(REPO, repo_type="dataset")
        if f.startswith(prefixe) and f.endswith(".parquet")
        and MOTIF_FICHIER.match(f.split("/")[-1])
    ]
    attends = [f for f in noms if f.split("/")[-1].startswith(f"{LANGUE}-{split}-")]
    if not attends:
        raise FileNotFoundError(
            f"Aucun parquet pour {REPO} / {LANGUE}_{TACHE} / {split}"
        )
    return sorted(attends)


def charger_split(split):
    """Charge un split via le builder parquet generique.

    La config `ewe_asr` declaree dans le YAML du dataset echoue au
    chargement (colonne `__index_level_0__` absente du schema declare) ;
    on lit donc les fichiers parquet directement.
    """
    urls = [f"hf://datasets/{REPO}@main/{f}" for f in fichiers_parquet(split)]
    return load_dataset("parquet", data_files=urls, split="train")


def duree_secondes(extrait):
    """Duree en secondes, lue dans les octets audio bruts (MP3)."""
    audio = extrait.get("audio")
    if not isinstance(audio, dict) or not audio.get("bytes"):
        return None
    try:
        infos = sf.info(io.BytesIO(audio["bytes"]))
    except (sf.LibsndfileError, RuntimeError):
        return None
    if not infos.samplerate:
        return None
    return infos.frames / infos.samplerate


def charger_audio_16k(extrait):
    """Decode le MP3, moyenne les canaux, reechantillonne a 16 kHz.

    Renvoie un tableau float32 1-D a 16 kHz, ou None si l'audio est
    illisible. Le format source est MP3 44,1 kHz stereo (verifie le
    2026-11-16).
    """
    audio = extrait.get("audio")
    if not isinstance(audio, dict) or not audio.get("bytes"):
        return None
    try:
        piste, frequence = sf.read(
            io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True
        )
    except (sf.LibsndfileError, RuntimeError):
        return None
    mono = piste.mean(axis=1)
    if frequence != 16000:
        mono = signal.resample_poly(mono, 16000, frequence)
    return mono.astype(np.float32)


def normaliser(texte):
    """Normalisation commune a toutes les metriques (documentee doc 14,
    section 7) : NFKC, minuscules, ponctuation retiree, espaces replies."""
    texte = unicodedata.normalize("NFKC", str(texte)).lower()
    texte = re.sub(r"[^\w\s]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte.strip()


def wer_cer(references, predictions):
    """WER et CER agreges sur le corpus (jiwer), apres normalisation."""
    paires = [(normaliser(r), normaliser(p)) for r, p in zip(references, predictions)]
    paires = [(r, p) for r, p in paires if r]
    refs, hyps = zip(*paires) if paires else ([], [])
    if not refs:
        return 1.0, 1.0
    return jiwer.wer(list(refs), list(hyps)), jiwer.cer(list(refs), list(hyps))


def wer_par_extrait(references, predictions):
    """WER par extrait (pour l'analyse des pires erreurs)."""
    scores = []
    for r, p in zip(references, predictions):
        rn = normaliser(r)
        if not rn:
            continue
        pn = normaliser(p) or " "
        scores.append(jiwer.wer(rn, pn))
    return scores




In [ ]:
# Chargement des trois splits (telechargement ~5,7 Go au total).
brut = {s: charger_split(s) for s in SPLITS}
for s, d in brut.items():
    print(f"{s:12s} {len(d):6d} extraits")




In [ ]:
# Statistiques par split : durees, heures, locuteurs.
# Enregistre aussi les manifestes complets dans resultats/.
stats = {}
for s in SPLITS:
    lignes = []
    t0 = time.time()
    for i, ex in enumerate(brut[s]):
        texte = str(ex.get("transcription") or "").strip()
        lignes.append({
            "id": ex.get("id"),
            "speaker_id": ex.get("speaker_id"),
            "gender": ex.get("gender"),
            "duree_s": duree_secondes(ex),
            "n_mots": len(texte.split()),
            "transcription": texte,
        })
        if (i + 1) % 1000 == 0:
            print(f"  {s} : {i + 1} extraits en {time.time() - t0:.0f} s")
    df = pd.DataFrame(lignes)
    df.to_csv(SORTIES / f"ewe_asr_{s}_manifeste.csv", index=False)
    stats[s] = df
    valides = df.dropna(subset=["duree_s"])
    heures = valides["duree_s"].sum() / 3600
    print(f"{s:12s} {len(df):6d} extraits | {heures:7.2f} h | "
          f"p50 {valides['duree_s'].median():5.1f} s | max {valides['duree_s'].max():5.1f} s | "
          f"{df['speaker_id'].nunique():4d} locuteurs | audio illisible : {df['duree_s'].isna().sum()}")




In [ ]:
# Orthographe et style de transcription (sur le train).
textes = stats["train"].dropna(subset=["transcription"])["transcription"]
df_train = stats["train"].dropna(subset=["duree_s"])

inventaire = Counter(c for t in textes for c in t if not c.isspace())
print("Caracteres distincts :", len(inventaire))
print("Plus frequents :", inventaire.most_common(15))
print("Lettres ewe presentes :", {c: inventaire.get(c, 0) for c in "ɖɛɔŋƒɣʋ"})
print("Extraits avec ponctuation forte (.,;:!?) :",
      f"{sum(any(c in '.,;:!?' for c in t) for t in textes) / len(textes):.0%}")
print("Extraits avec majuscules :",
      f"{sum(any(c.isupper() for c in t) for t in textes) / len(textes):.0%}")

debit = df_train["n_mots"] / df_train["duree_s"]
print(f"Debit (mots/s) : p05 {debit.quantile(0.05):.2f} | p50 {debit.median():.2f} | p95 {debit.quantile(0.95):.2f}")
print(f"Clips > 30 s : {(df_train['duree_s'] > 30).sum()} ({(df_train['duree_s'] > 30).mean():.1%})")
print(f"Clips < 1,5 s : {(df_train['duree_s'] < 1.5).sum()}")
print(f"Champ gender renseigne : {(df_train['gender'].notna() & (df_train['gender'] != '')).sum()} / {len(df_train)}")
print()
print("Exemples de transcriptions :")
for t in textes.head(3):
    print("-", t[:150])




## Lecture des statistiques

Points à relever avant l'entraînement (notebook 07) :

- **Heures conservées** par split : c'est le budget réel d'entraînement.
- **Clips > 30 s** : le v0.1 les écarte (limite d'entrée Whisper) ; le taux est reporté ici pour documenter la perte.
- **Style des transcriptions** : si elles sont casées et ponctuées (attendu), Whisper les apprendra telles quelles ; si elles sont en minuscules brutes, le modèle produira du texte sans casse.
- **Genre** : le champ était vide sur l'échantillon local du 2026-11-16 ; cette cellule confirme sur le corpus complet.




In [ ]:
# Preparation du jeu d'evaluation : test complet (ou sous-echantillon).
# Le decodage MP3 -> mono 16 kHz est fait une fois pour les deux baselines.
MAX_EVAL = None  # ex. : 300 pour un premier passage rapide

references = [str(t or "").strip() for t in brut["test"]["transcription"]]
if MAX_EVAL:
    references = references[:MAX_EVAL]

audios_eval = []
refs_eval = []
echecs = 0
t0 = time.time()
for ex, ref in zip(brut["test"], references):
    audio = charger_audio_16k(ex)
    if audio is None:
        echecs += 1
        continue
    audios_eval.append(audio)
    refs_eval.append(ref)
    if (len(audios_eval)) % 500 == 0:
        print(f"  {len(audios_eval)} audios decodes en {time.time() - t0:.0f} s")
print(f"Evaluation sur {len(audios_eval)} extraits "
      f"({len(references) - len(audios_eval)} audio illisible) | "
      f"{sum(len(a) / 16000 for a in audios_eval) / 3600:.2f} h d'audio")




In [ ]:
# Baseline 1 : MMS-1B zero-shot (Meta), adaptateur ewe.
# Licence CC-BY-NC-4.0 : evaluation uniquement, jamais dans un modele publie.
from transformers import AutoProcessor, Wav2Vec2ForCTC

MODELE_MMS = "facebook/mms-1b-all"
LANGUE_MMS = "ewe"  # code ISO 639-3 attendu

processor_mms = AutoProcessor.from_pretrained(MODELE_MMS, target_language=LANGUE_MMS)
modele_mms = Wav2Vec2ForCTC.from_pretrained(
    MODELE_MMS, target_language=LANGUE_MMS, ignore_mismatched_sizes=True
).to(DEVICE)

langues = getattr(modele_mms, "supported_languages", None) or \
    getattr(getattr(modele_mms, "config", None), "supported_languages", None)
if langues and LANGUE_MMS not in langues:
    print("Langues proches :", sorted(l for l in langues if l.startswith("e")))
    raise ValueError(f"Code langue MMS inconnu : {LANGUE_MMS}")
print("MMS pret, adaptateur :", LANGUE_MMS)




In [ ]:
# Evaluation MMS : la plus longue (~45-70 min sur le test complet, T4).
def evaluer_mms(audios, taille_lot=4):
    modele_mms.eval()
    predictions = []
    t0 = time.time()
    for i in range(0, len(audios), taille_lot):
        lot = audios[i:i + taille_lot]
        entrees = processor_mms(
            lot, sampling_rate=16000, return_tensors="pt", padding=True
        ).to(DEVICE)
        with torch.no_grad():
            logits = modele_mms(**entrees).logits
        ids = torch.argmax(logits, dim=-1)
        predictions.extend(processor_mms.batch_decode(ids))
        if (i // taille_lot) % 25 == 0:
            print(f"  {min(i + taille_lot, len(audios))}/{len(audios)} "
                  f"en {time.time() - t0:.0f} s", end="\r")
    print()
    return predictions

predictions_mms = evaluer_mms(audios_eval)
wer_mms, cer_mms = wer_cer(refs_eval, predictions_mms)
print(f"MMS zero-shot : WER {wer_mms:.1%} | CER {cer_mms:.1%}")

# Liberer la memoire avant la baseline suivante.
del modele_mms, processor_mms
torch.cuda.empty_cache()




In [ ]:
# Baseline 2 : WAXALNet (whisper-small fine-tune sur WAXAL ewe, publique).
from transformers import pipeline

MODELE_WAXALNET = "waxal-benchmarking/whisper-small-waxal-ewe"
asr = pipeline(
    "automatic-speech-recognition",
    model=MODELE_WAXALNET,
    device=0 if DEVICE == "cuda" else -1,
    chunk_length_s=30,
)

predictions_waxalnet = []
t0 = time.time()
TAILLE_LOT = 8
for i in range(0, len(audios_eval), TAILLE_LOT):
    lot = audios_eval[i:i + TAILLE_LOT]
    sorties = asr(lot, batch_size=TAILLE_LOT)
    predictions_waxalnet.extend(p["text"].strip() for p in sorties)
    if (i // TAILLE_LOT) % 25 == 0:
        print(f"  {min(i + TAILLE_LOT, len(audios_eval))}/{len(audios_eval)} "
              f"en {time.time() - t0:.0f} s", end="\r")
print()

wer_waxalnet, cer_waxalnet = wer_cer(refs_eval, predictions_waxalnet)
print(f"WAXALNet : WER {wer_waxalnet:.1%} | CER {cer_waxalnet:.1%}")
print("(reference annoncee sur leur sous-ensemble prive filtre : WER 32,3 / CER 10,6)")




In [ ]:
# Synthese des baselines (fichier lu ensuite par le notebook 07).
resultats = pd.DataFrame([
    {"modele": "MMS-1B zero-shot (adaptateur ewe)",
     "wer": wer_mms, "cer": cer_mms, "n": len(refs_eval)},
    {"modele": "WAXALNet whisper-small (public)",
     "wer": wer_waxalnet, "cer": cer_waxalnet, "n": len(refs_eval)},
])
resultats.to_json(SORTIES / "baselines_test.json", orient="records", indent=2)
resultats




In [ ]:
# Exemples cote a cote (3 premiers extraits) pour l'analyse qualitative.
for i in range(3):
    print("REF :", refs_eval[i][:120])
    print("MMS :", predictions_mms[i][:120])
    print("WXN :", predictions_waxalnet[i][:120])
    print()




## Conclusion du notebook 06

- Les scores ci-dessus sont la référence à battre par notre fine-tuning (notebook 07).
- Le WER 32,3 de WAXALNet a été mesuré sur un sous-ensemble privé (filtre non reproduisible, voir doc 14, section 7) : notre évaluation sur le test public complet est la comparaison honnête.
- MMS est CC-BY-NC-4.0 : il ne sert que de point de comparaison zero-shot.
- **Téléchargez `resultats/` avant de fermer la VM Colab** (notamment `baselines_test.json`), puis passez au notebook 07.


